# SC16 - Detection Model Benchmark
Trains several detectors under **identical settings** (same data split, imgsz, epochs, seed) so the
comparison is fair, then reports P / R / mAP@50 / mAP@50-95 / F1 / params / GFLOPs / speed and
exports each model's **own** PR / F1-conf / P-conf / R-conf / confusion / training-curve PNGs
(one model per file). Goal: show **YOLO11** is the best trade-off for this dataset.

**Attach:** `thesis-common-10c` + the raw image dataset. GPU on.

In [ ]:
import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| threat levels:', T.NUM_THREAT)

# generic finders across kaggle inputs + local Results
ROOTS = ['/kaggle/input', 'Results', '.']
def find(fn):
    for r in ROOTS:
        h = sorted(glob.glob(f'{r}/**/{fn}', recursive=True), key=len)
        if h: return h[0]
    return None
def find_all(pat):
    seen = {}
    for r in ROOTS:
        for p in sorted(glob.glob(f'{r}/**/{pat}', recursive=True), key=len):
            seen.setdefault(os.path.basename(p), p)
    return list(seen.values())
OUT = '/kaggle/working'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# ---- build the YOLO dataset once (same export the main pipeline uses) ----
import subprocess, sys, shutil, time
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics'], check=False)
from ultralytics import YOLO
try:
    from ultralytics import RTDETR
except Exception:
    RTDETR = None
ROOT = T.find_image_root()
DATA_YAML = T.export_yolo_dataset(ROOT, '/kaggle/working/yolo_ds', include_aug=False)
print('data.yaml ->', DATA_YAML)

# identical training budget for every detector
DET_EPOCHS = 60      # raise if you have time budget; keep equal across models for fairness
IMGSZ      = 640
BATCH      = 16
SEED       = getattr(T, 'SEED', 42)

In [ ]:
# ---- the detector zoo. Same-scale 's' models + one transformer detector (RT-DETR).  ----
# Any weight that fails to download is skipped so the notebook still finishes.
ZOO = [
    ('yolov8s',  'YOLO',   'yolov8s.pt'),
    ('yolov9s',  'YOLO',   'yolov9s.pt'),
    ('yolov10s', 'YOLO',   'yolov10s.pt'),
    ('yolo11s',  'YOLO',   'yolo11s.pt'),   # <-- ours
    ('yolo12s',  'YOLO',   'yolo12s.pt'),
    ('rtdetr-l', 'RTDETR', 'rtdetr-l.pt'),  # transformer detector, for contrast
]

def build(kind, weights):
    if kind == 'RTDETR':
        if RTDETR is None: raise RuntimeError('RTDETR unavailable in this ultralytics build')
        return RTDETR(weights)
    return YOLO(weights)

def n_params(m):
    try:   return sum(p.numel() for p in m.model.parameters())
    except Exception: return float('nan')
def gflops(m):
    try:
        info = m.model.info(verbose=False)   # (layers, params, grads, flops)
        return float(info[3])
    except Exception: return float('nan')

In [ ]:
import pandas as pd
DET_DIR = os.path.join(OUT, 'det_bench'); os.makedirs(DET_DIR, exist_ok=True)
CURVES = ['PR_curve.png','F1_curve.png','P_curve.png','R_curve.png',
          'confusion_matrix.png','confusion_matrix_normalized.png']
rows = []
for name, kind, weights in ZOO:
    print('\n=============================', name)
    try:
        m = build(kind, weights)
    except Exception as e:
        print('  skip (load failed):', e); continue
    par, fl = n_params(m), gflops(m)
    try:
        m.train(data=DATA_YAML, epochs=DET_EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
                project=OUT, name=f'train_{name}', exist_ok=True, verbose=False, plots=True)
        r = m.val(data=DATA_YAML, split='test', project=OUT, name=f'val_{name}',
                  exist_ok=True, verbose=False, plots=True)
    except Exception as e:
        print('  skip (train/val failed):', e); continue
    P, R = float(r.box.mp), float(r.box.mr)
    f1 = (2*P*R/(P+R)) if (P+R) else 0.0
    spd = getattr(r, 'speed', {}) or {}
    ms = float(spd.get('inference', float('nan')))
    rows.append({'model':name, 'type':('Transformer' if kind=='RTDETR' else 'CNN'),
                 'precision':P, 'recall':R, 'f1':f1,
                 'map50':float(r.box.map50), 'map50_95':float(r.box.map),
                 'params_M':par/1e6 if par==par else float('nan'),
                 'GFLOPs':fl, 'infer_ms':ms})
    # ---- copy each model's own curves (already single-model PNGs) ----
    dst = os.path.join(DET_DIR, name); os.makedirs(dst, exist_ok=True)
    vdir = os.path.join(OUT, f'val_{name}'); tdir = os.path.join(OUT, f'train_{name}')
    for c in CURVES:
        s = os.path.join(vdir, c)
        if os.path.exists(s): shutil.copy(s, os.path.join(dst, f'{name}_{c}'))
    for extra in ['results.png','results.csv']:      # training curves
        s = os.path.join(tdir, extra)
        if os.path.exists(s): shutil.copy(s, os.path.join(dst, f'{name}_{extra}'))
    for pb in sorted(glob.glob(os.path.join(vdir,'val_batch*_pred.jpg')))[:1]:
        shutil.copy(pb, os.path.join(dst, f'{name}_pred_sample.jpg'))
    print('  done:', {k:round(v,4) for k,v in rows[-1].items() if isinstance(v,float)})

In [ ]:
# ---- comparison table (CSV + JSON + LaTeX) ----
df = pd.DataFrame(rows).set_index('model')
order = ['type','precision','recall','f1','map50','map50_95','params_M','GFLOPs','infer_ms']
df = df[order].sort_values('map50_95', ascending=False)
df.to_csv(os.path.join(OUT,'detection_comparison.csv'))
json.dump(df.reset_index().to_dict('records'), open(os.path.join(OUT,'detection_comparison.json'),'w'), indent=2)
disp = df.copy()
for c in ['precision','recall','f1','map50','map50_95']: disp[c]=disp[c].map(lambda v:f'{v:.3f}')
for c in ['params_M','GFLOPs','infer_ms']: disp[c]=disp[c].map(lambda v:f'{v:.1f}')
print(disp)

def to_latex(df, caption, label, headers):
    cols = list(df.columns)
    lines = [r'\begin{table}[t]', r'\centering', f'\\caption{{{caption}}}', f'\\label{{{label}}}',
             '\\begin{tabular}{l'+'c'*len(cols)+'}', r'\toprule',
             'Model & '+' & '.join(headers)+r' \\', r'\midrule']
    best = df['map50_95'].astype(float).idxmax() if 'map50_95' in df else None
    for idx,row in df.iterrows():
        nm = idx.replace('_','\\_')
        if idx==best: nm = r'\textbf{'+nm+'}'
        lines.append(nm+' & '+' & '.join(str(row[c]) for c in cols)+r' \\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)

tex = to_latex(disp, 'Detection model comparison on the held-out test split (identical training budget).',
               'tab:det_compare',
               ['Type','P','R','F1','mAP@50','mAP@50:95','Params (M)','GFLOPs','Speed (ms)'])
open(os.path.join(OUT,'detection_comparison.tex'),'w').write(tex)
print('\n', tex)
print('\nPer-model curves in det_bench/<model>/  (one model per PNG).')